# Group 05, Project 1, Deliverable 3: EDA, Statistical Inference and Dashboard

**Course:** ACIT 4880, CRN 45883, Set B, Domain D (tariffs and trade)
**Members:** Jashanpreet Singh, Priyanshu Batra, Janek Basi

**Business question (from D1, unchanged):** how did US imports of capital goods, automotive goods and consumer goods move through the 2025 tariff period, and how much of that movement came from price versus volume? Stakeholder: procurement and supply chain teams at US importers.

**Target variable:** `import_value_usd`, the monthly US general import value in US dollars for one partner country and one 5 digit end-use product code.

**Which version of the data we use.** Everything below runs on `data/processed/d2_clean_pre_transform.csv.gz`. This file is saved at the end of D2 Part 4, so it already has the type fixes, duplicate removal, domain fixes, imputation and outlier handling from D2, but none of the log transforms or scaling from D2 Part 5. The transformed version stays for Project 2. The file has 277,592 rows and 22 columns.

We add four readable columns from values that are already in the data (no transformation):
- `effective_tariff_rate` = calculated duty / consumption value, so the share of the goods value that was paid as duty
- `period` = pre-tariff (Jan 2022 to Mar 2025, 39 months) vs post-tariff (Apr 2025 on, 16 months). April 2025 is when the broad "reciprocal" tariffs started.
- `pays_duty` = whether the flow paid any duty at all
- `main_mode` = the transport mode carrying the larger share of the value (vessel, air, or land and other)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.colors import LogNorm
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}" if abs(v) < 1000 else f"{v:,.0f}")
sns.set_theme(style="whitegrid")

# Cleaned D2 data, saved after outlier handling (Part 4) and before any log/sqrt transform (Part 5)
df = pd.read_csv("data/processed/d2_clean_pre_transform.csv.gz", parse_dates=["month"],
                 dtype={"cty_code": str, "enduse_code": str, "category_code": str})
TARGET = "import_value_usd"

# A few readable columns built from the raw values (no transformation involved)
df["effective_tariff_rate"] = np.where(df["consumption_value_usd"] > 0,
                                       df["calculated_duty_usd"] / df["consumption_value_usd"], 0.0)
df["period"] = np.where(df["month"] >= "2025-04-01", "Post-tariff (Apr 2025 on)", "Pre-tariff (Jan 2022 to Mar 2025)")
df["pays_duty"] = np.where(df["calculated_duty_usd"] > 0, "Pays duty", "Duty free")
df["main_mode"] = np.select(
    [(df["vessel_value_usd"] >= df["air_value_usd"]) & (df["vessel_value_usd"] > 0), df["air_value_usd"] > 0],
    ["Vessel", "Air"], default="Land and other")

CONT = ["import_value_usd", "consumption_value_usd", "dutiable_value_usd", "calculated_duty_usd",
        "air_value_usd", "vessel_value_usd", "container_value_usd", "air_weight_kg", "vessel_weight_kg",
        "price_index", "effective_tariff_rate"]
def fmt_p(p):
    # with n in the hundreds of thousands, p can be smaller than the smallest float and print as 0
    return "< 1e-300 (too small to store as a float, effectively 0)" if p == 0 else f"{p:.3g}"

print(df.shape)
df.head()

(277592, 26)


,cty_code,country,enduse_code,enduse_desc,import_value_usd,consumption_value_usd,dutiable_value_usd,calculated_duty_usd,air_value_usd,vessel_value_usd,container_value_usd,air_weight_kg,vessel_weight_kg,category_code,month,fred_series,price_index,category,price_index_missing,air_weight_kg_missing,vessel_weight_kg_missing,is_outlier_import_value,effective_tariff_rate,period,pays_duty,main_mode
0,1220,Canada,20000,"Generators, Accessories",107499386,107923317,6909147,270810,7705117,46471,46471,"27,729","3,029",2,2022-01-01,IR2,91.4000,Capital goods,0,0,0,1,0.0025,Pre-tariff (Jan 2022 to Mar 2025),Pays duty,Air
1,2010,Mexico,20000,"Generators, Accessories",621390284,617165194,59181356,2013272,12922581,452706,452706,"32,510","56,175",2,2022-01-01,IR2,91.4000,Capital goods,0,0,0,1,0.0033,Pre-tariff (Jan 2022 to Mar 2025),Pays duty,Air
2,2050,Guatemala,20000,"Generators, Accessories",109662,109662,0,0,109662,0,0,104.0000,0.0000,2,2022-01-01,IR2,91.4000,Capital goods,0,0,0,0,0.0000,Pre-tariff (Jan 2022 to Mar 2025),Duty free,Air
3,2150,Honduras,20000,"Generators, Accessories",149930,149930,149930,3702,149930,0,0,"6,056",0.0000,2,2022-01-01,IR2,91.4000,Capital goods,0,0,0,0,0.0247,Pre-tariff (Jan 2022 to Mar 2025),Pays duty,Air
4,2230,Costa Rica,20000,"Generators, Accessories",1284680,1284680,221633,6741,1192746,54631,54631,"11,976",874.0000,2,2022-01-01,IR2,91.4000,Capital goods,0,0,0,0,0.0052,Pre-tariff (Jan 2022 to Mar 2025),Pays duty,Air


# Part 2: Statistical Inference

Every test answers a question that came out of Part 1, and every test uses α = 0.05.

**A note on sample size.** We have 277,592 rows, so even tiny differences come out as "significant". Several p-values are so small that Python cannot store them and they underflow to 0, so we print them as "< 1e-300". To keep the conclusions honest we also report an effect size (Cohen's d, Cohen's h or Cramér's V) and say in words whether the difference is big enough to matter for an importer, not just whether it is significant.

## 2.1 Confidence interval for a mean

**Question from the EDA:** the boxplot in 1.4 showed the effective tariff rate jumped after April 2025. What is the average rate an importer actually paid in the post-tariff period? This is central to the business question because it is the cost the procurement team has to plan for.

In [2]:
x = df.loc[df["period"] == "Post-tariff (Apr 2025 on)", "effective_tariff_rate"]
n = len(x); xbar = x.mean(); s = x.std(ddof=1); dof = n - 1
t_star = stats.t.ppf(0.975, dof)
se = s / np.sqrt(n); me = t_star * se
steps = pd.Series({"sample mean (x bar)": xbar, "standard deviation (s)": s, "sample size (n)": n,
                   "degrees of freedom (n - 1)": dof, "critical value t* (95%)": t_star,
                   "standard error (s / sqrt n)": se, "margin of error (t* x SE)": me,
                   "lower bound": xbar - me, "upper bound": xbar + me}, name="value")
display(steps.to_frame())
print(f"95% CI for the mean effective tariff rate after April 2025: {xbar - me:.4%} to {xbar + me:.4%}")

,value
sample mean (x bar),0.1334
standard deviation (s),0.1104
sample size (n),"79,967"
degrees of freedom (n - 1),"79,966"
critical value t* (95%),1.9600
standard error (s / sqrt n),0.0004
margin of error (t* x SE),0.0008
lower bound,0.1327
upper bound,0.1342


95% CI for the mean effective tariff rate after April 2025: 13.2659% to 13.4190%


**Steps:** x̄ = 0.1334, s = 0.1104, n = 79,967, df = 79,966, t* = 1.960 (with this many degrees of freedom the t value is the same as z to three decimals), SE = 0.1104 / √79,967 = 0.00039, margin of error = 1.960 × 0.00039 = 0.00077.

**Interval:** 13.27% to 13.42%.

**In plain words:** we are 95% confident that, after April 2025, a typical US import flow in these three categories paid on average between 13.3% and 13.4% of its value in duty. That is about 5.6 times the 2.38% average before the tariffs. The interval is very narrow because n is so large, so the uncertainty is about 0.08 of a percentage point.